# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv(r"C:\Users\Manish\OneDrive\Desktop\New folder\week7\netflix_catalogue.csv")
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 8807 titles
type
Movie      6131
TV Show    2676
Name: count, dtype: int64
  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water              NaN   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans              NaN   
4      s5  TV Show           Kota Factory              NaN   

                                                cast        country  \
0                                                NaN  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...            NaN   
3                                                NaN            NaN   
4  Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...          India   

           date_added  release_year rating   duration  \
0  September 25, 2021          2020  PG-13     90 min   
1  

In [11]:
print("Top Genres:")
print(df['listed_in'].value_counts().head(8))

print("\nTop Countries:")
print(df['country'].value_counts().head(8))

print("\nRatings:")
print(df['rating'].value_counts())

Top Genres:
listed_in
Dramas, International Movies                        362
Documentaries                                       359
Stand-Up Comedy                                     334
Comedies, Dramas, International Movies              274
Dramas, Independent Movies, International Movies    252
Kids' TV                                            220
Children & Family Movies                            215
Children & Family Movies, Comedies                  201
Name: count, dtype: int64

Top Countries:
country
United States     2818
India              972
United Kingdom     419
Japan              245
South Korea        199
Canada             181
Spain              145
France             124
Name: count, dtype: int64

Ratings:
rating
TV-MA       3207
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
NC-17          3
UR             3
74 min         

## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [10]:
# Task 1
# YOUR CODE HERE
import pandas as pd
import plotly.express as px

# Load dataset
df = pd.read_csv(r"C:\Users\Manish\OneDrive\Desktop\New folder\week7\netflix_catalogue.csv")

# Create decade column
df["decade"] = (df["release_year"] // 10 * 10).astype(str) + "s"

# Filter to common ratings
common_ratings = ["TV-14", "TV-MA", "PG-13", "R", "PG"]
filtered_df = df[df["rating"].isin(common_ratings)]

# Count titles by rating and decade
heatmap_data = (
    filtered_df.groupby(["rating", "decade"])
    .size()
    .reset_index(name="count")
)

# Plot heatmap
fig = px.imshow(
    heatmap_data.pivot(index="rating", columns="decade", values="count"),
    color_continuous_scale="Blues",
    text_auto=True,
    labels=dict(x="Decade", y="Rating", color="Number of Titles"),
    title="Heatmap of Netflix Titles by Rating and Release Decade"
)
fig.show()


## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [9]:
# Task 2
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go

# Load dataset
df = pd.read_csv(
    r"C:\Users\Manish\OneDrive\Desktop\New folder\week7\netflix_catalogue.csv"
)

# Convert date_added to datetime and create added_year
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['added_year'] = df['date_added'].dt.year

# Filter Movies only
movies_df = df[df['type'] == 'Movie']

# Count movies added per year
yearly_additions = (
    movies_df.groupby('added_year')
    .size()
    .reset_index(name='additions')
    .sort_values('added_year')
)

# Calculate cumulative total
yearly_additions['cumulative_total'] = yearly_additions['additions'].cumsum()

# Find year with largest addition
max_year = yearly_additions.loc[
    yearly_additions['additions'].idxmax(),
    'added_year'
]

max_value = yearly_additions['additions'].max()

# Create waterfall chart
fig = go.Figure()

fig.add_trace(
    go.Waterfall(
        name="Movie Additions",
        orientation="v",
        measure=["relative"] * len(yearly_additions) + ["total"],
        x=list(yearly_additions["added_year"]) + ["Total"],
        y=list(yearly_additions["additions"]) +
          [yearly_additions["cumulative_total"].iloc[-1]],
        textposition="outside",
        connector={"line": {"color": "rgba(63,63,63,0.5)"}},
        increasing={"marker": {"color": "green"}},
        totals={"marker": {"color": "blue"}}
    )
)

# Annotation for largest addition year
fig.add_annotation(
    x=max_year,
    y=max_value,
    text=f"Largest addition: {int(max_year)} ({max_value} titles)",
    showarrow=True,
    arrowhead=2,
    ax=0,
    ay=-40
)

# Layout
fig.update_layout(
    title="Netflix Movie Library Growth (2015–2022)",
    xaxis_title="Year Added",
    yaxis_title="Number of Movies Added",
    waterfallgap=0.3
)

# Display chart
fig.show()

# If fig.show() gives Plotly renderer error, use:
# fig.write_html("netflix_movie_growth.html")
fig.show()
